# Cálculo de emissões por atividade produtiva

Esse notebook faz o cálculo das emissões por atividade produtiva.

## 1. Inicialização

Carrega dependências e executa configurações iniciais.

Salve o notebook e execute todas as células em um kernel novo para gerar outputs rastreáveis. O registro usa o código salvo em disco e não captura edições não salvas nem alterações manuais de variáveis.

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

from scr import arquivos, mip, modelos, emissoes
from scr.proveniencia import Execucao

execucao = Execucao("analise_matriz_insumo_produto.ipynb")

sns.set_theme(context="notebook", style="whitegrid")

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 70)

## 2. Carregamento dos dados de entrada

### 2.1. MIP brasileira de 2015

Carrega a MIP brasileira de 2015, verifica a integridade do arquivo e apresenta as primeiras linhas dos coeficientes técnicos da Tabela 14.
Fonte: IBGE.

In [15]:
tabelas = execucao.carregar(
    arquivos.carregar_matriz_67, arquivos.ARQUIVO_MIP_67,
    "MIP brasileira de 2015, 67 atividades (IBGE).",
)
mip.carregar_coeficientes_tecnicos_67(tabelas).head().iloc[:, :5]

Checksum validado: Matriz_de_Insumo_Produto_2015_Nivel_67.xls


atividade_destino,0191,0192,0280,0580,0680
atividade_origem,,,,,
0191,0.021056,0.027488,0.004699,0.000106,0.000032
0192,0.002413,0.032509,0.003979,0.000254,0.000086
0280,0.002699,0.007325,0.048358,0.000118,0.000007
0580,0.000514,0.002551,0.000358,0.015136,0.002867
0680,0.000020,0.000029,0.000010,0.000062,0.070423


### 2.2. Intensidades de CO₂ do Brasil

Carrega as intensidades de emissão de CO₂ do Brasil para 2011 e 2018, em Gg de CO₂ por R$ milhão de produção bruta, verifica a integridade do arquivo e apresenta os primeiros setores.
Fonte: Sanguinet e Azzoni (2024).


In [16]:
matriz_coeficientes_co2 = execucao.carregar(
    arquivos.carregar_coeficientes_co2, arquivos.ARQUIVO_COEFICIENTES_CO2,
    "Intensidades de CO₂ do Brasil, 2011 e 2018 (Sanguinet e Azzoni, 2024).",
)
matriz_coeficientes_co2.head()

Checksum validado: coeficientes_co2_2011_2018.csv


ano,2011,2018
setor_id,,
S1,0.04,0.03
S2,0.05,0.04
S3,0.19,0.17
S4,0.07,0.08
S5,0.01,0.01


### 2.3. Parâmetros do exterior representativo

Carrega as intensidades de emissão de CO₂ e a inversa de Leontief do exterior representativo, verifica a integridade dos dois arquivos por SHA-256 e apresenta os primeiros setores de cada matriz.

Neste cenário, assume-se que o exterior possui as mesmas intensidades de emissões e tecnologia que o Brasil.

Fonte: ver fontes em 2.1 e 2.2.


In [17]:
intensidades_co2_exterior = execucao.carregar(
    arquivos.carregar_intensidades_co2_exterior_representativo,
    arquivos.ARQUIVO_INTENSIDADES_CO2_EXTERIOR,
    "Intensidades de CO₂ do exterior representativo, por ano de cenário.",
)
inversa_leontief_exterior = execucao.carregar(
    arquivos.carregar_inversa_leontief_exterior_representativo,
    arquivos.ARQUIVO_INVERSA_LEONTIEF_EXTERIOR,
    "Inversa de Leontief do exterior representativo (parâmetro de cenário).",
)

display(intensidades_co2_exterior.head())
display(inversa_leontief_exterior.head().iloc[:, :5])

ano,2011,2018
atividade,,
0191,0.04,0.03
0192,0.05,0.04
0280,0.19,0.17
0580,0.07,0.08
0680,0.01,0.01


atividade_destino,0191,0192,0280,0580,0680
atividade_origem,,,,,
0191,1.027732,0.054436,0.013645,0.007712,0.004759
0192,0.003118,1.040622,0.005558,0.000849,0.000472
0280,0.003578,0.009131,1.051213,0.000648,0.000400
0580,0.003023,0.004454,0.000884,1.016587,0.003983
0680,0.022982,0.019166,0.010885,0.030039,1.088226


## 3. Construir os coeficientes técnicos de Leontief

A Tabela 11 fornece `Bn`, os coeficientes de insumos nacionais (produto × atividade, 127 × 67). A Tabela 13 fornece `D`, a participação setorial da produção nacional (atividade × produto, 67 × 127). A composição `A = D @ Bn` produz a matriz de coeficientes técnicos intersetoriais (atividade × atividade). Em `A`, cada coluna `j` é dividida pela produção bruta da atividade `j`.

In [4]:
matriz_bn = mip.carregar_matriz_bn_67(tabelas)
matriz_d = mip.carregar_matriz_participacao_67(tabelas)
coeficientes_tecnicos = modelos.calcular_coeficientes_tecnicos_67(tabelas)
print(f"Bn (produto × atividade): {matriz_bn.shape}")
print(f"D (atividade × produto): {matriz_d.shape}")
coeficientes_tecnicos

Bn (produto × atividade): (127, 67)
D (atividade × produto): (67, 127)


atividade_destino,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,2.105648e-02,2.748793e-02,4.699289e-03,0.000106,3.249443e-05,0.000011,...,0.000467,0.001145,0.000927,0.000081,1.517130e-03,0.0
0192,2.413068e-03,3.250905e-02,3.978985e-03,0.000254,8.591947e-05,0.000017,...,0.000131,0.000422,0.000512,0.000080,2.369667e-04,0.0
0280,2.698819e-03,7.324610e-03,4.835818e-02,0.000118,6.716297e-06,0.000002,...,0.000038,0.000173,0.000060,0.000010,1.901811e-05,0.0
0580,5.143662e-04,2.550859e-03,3.580135e-04,0.015136,2.866503e-03,0.000011,...,0.000012,0.000037,0.000027,0.000033,1.987736e-05,0.0
0680,2.042381e-05,2.938643e-05,9.615380e-06,0.000062,7.042309e-02,0.001994,...,0.000090,0.000007,0.000021,0.000268,9.124351e-05,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8691,3.284214e-08,3.103543e-08,4.210832e-08,0.000002,2.623695e-07,0.000066,...,0.000001,0.000002,0.000896,0.000001,8.089819e-07,0.0
8692,6.661799e-08,1.074156e-07,6.357413e-07,0.000003,5.476924e-06,0.000005,...,0.000112,0.000080,0.093559,0.000334,5.721761e-05,0.0
9080,5.390155e-06,7.548206e-05,7.115275e-05,0.000048,7.933534e-05,0.000008,...,0.000640,0.000504,0.000025,0.023168,9.120232e-03,0.0


### Verificação da matriz A

A Tabela 14 (`D.Bn`) é a publicação do IBGE para a mesma composição.

In [5]:
coeficientes_tecnicos_ibge = mip.carregar_coeficientes_tecnicos_67(tabelas)
erro_a = (coeficientes_tecnicos - coeficientes_tecnicos_ibge).abs().to_numpy().max()
print(f"Erro absoluto máximo em relação à Tabela 14 do IBGE: {erro_a:.2e}")
assert erro_a < 1e-10, "A matriz A calculada diverge da Tabela 14 do IBGE."

Erro absoluto máximo em relação à Tabela 14 do IBGE: 5.55e-17


## 4. Inversa de Leontief

Para uma demanda final `f`, o modelo aberto é `(I − A)x = f`. Portanto, `x = Lf`, onde `L = (I − A)⁻¹` é a inversa de Leontief ou matriz de requerimentos totais.

In [6]:
inversa_leontief = modelos.calcular_inversa_leontief(coeficientes_tecnicos)
inversa_leontief

atividade_destino,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,1.027732,0.054436,0.013645,0.007712,0.004759,0.004920,...,0.002084,0.005709,0.004749,0.002088,0.008247,0.0
0192,0.003118,1.040622,0.005558,0.000849,0.000472,0.000402,...,0.001097,0.003206,0.002639,0.000536,0.003484,0.0
0280,0.003578,0.009131,1.051213,0.000648,0.000400,0.000482,...,0.000384,0.000683,0.000783,0.000437,0.000788,0.0
0580,0.003023,0.004454,0.000884,1.016587,0.003983,0.000898,...,0.000296,0.000731,0.000422,0.000523,0.000626,0.0
0680,0.022982,0.019166,0.010885,0.030039,1.088226,0.032999,...,0.004184,0.003928,0.003700,0.005477,0.008738,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8691,0.000010,0.000010,0.000003,0.000011,0.000010,0.000076,...,0.000004,1.000011,0.000996,0.000006,0.000006,0.0
8692,0.000019,0.000022,0.000011,0.000027,0.000031,0.000033,...,0.000138,0.000107,1.103236,0.000397,0.000094,0.0
9080,0.000288,0.000526,0.000295,0.000479,0.000539,0.000440,...,0.001462,0.000989,0.000426,1.025167,0.010022,0.0


In [7]:
inversa_leontief_ibge = mip.carregar_inversa_leontief_ibge_67(tabelas)
erro_l = (inversa_leontief - inversa_leontief_ibge).abs().to_numpy().max()
print(f"Erro absoluto máximo em relação à Tabela 15 do IBGE: {erro_l:.2e}")
assert erro_l < 1e-10, "A inversa calculada diverge da matriz publicada pelo IBGE."

Erro absoluto máximo em relação à Tabela 15 do IBGE: 2.22e-15


## 5. Coeficientes e inversa de Ghosh

O modelo de Ghosh usa os mesmos fluxos intersetoriais, mas normaliza as **linhas**: `B = diag(x)⁻¹ @ Z`. Aqui, `Z = D @ U` usa os usos intermediários nacionais `U` da Tabela 03, e `x` é a produção bruta por atividade obtida da Tabela 01. Cada linha de `B` informa como a produção da atividade fornecedora é alocada entre as atividades demandantes. A inversa de Ghosh é `G = (I − B)⁻¹`.

In [8]:
matriz_transacoes = modelos.calcular_matriz_transacoes_intersetoriais_67(tabelas)
producao_bruta = modelos.calcular_producao_bruta_67(tabelas)
coeficientes_ghosh = modelos.calcular_coeficientes_alocacao_ghosh_67(tabelas)
print(f"Z (atividade × atividade): {matriz_transacoes.shape}")
print(f"Produção bruta: {producao_bruta.shape}")
coeficientes_ghosh

Z (atividade × atividade): (67, 67)
Produção bruta: (67,)


atividade_destino,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,2.105648e-02,1.217695e-02,4.924286e-04,6.733093e-06,1.806823e-05,0.000002,...,1.674059e-04,0.000663,0.000667,9.050670e-06,7.141232e-04,0.0
0192,5.447199e-03,3.250905e-02,9.412113e-04,3.659696e-05,1.078455e-04,0.000006,...,1.063183e-04,0.000552,0.000833,2.030040e-05,2.517916e-04,0.0
0280,2.575507e-02,3.096490e-02,4.835818e-02,7.177329e-05,3.563900e-05,0.000003,...,1.307821e-04,0.000957,0.000410,1.052095e-05,8.542922e-05,0.0
0580,8.062331e-03,1.771214e-02,5.880290e-04,1.513644e-02,2.498316e-02,0.000030,...,6.625472e-05,0.000340,0.000304,5.804098e-05,1.466551e-04,0.0
0680,3.673077e-05,2.341189e-05,1.812053e-06,7.118646e-06,7.042309e-02,0.000612,...,5.776687e-05,0.000007,0.000027,5.390115e-05,7.724057e-05,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8691,5.670011e-08,2.373594e-08,7.617833e-09,2.565959e-07,2.518677e-07,0.000020,...,6.693343e-07,0.000002,0.001114,2.442697e-07,6.574177e-07,0.0
8692,9.253152e-08,6.609394e-08,9.253152e-08,2.720856e-07,4.230012e-06,0.000001,...,5.569847e-05,0.000064,0.093559,5.191018e-05,3.740917e-05,0.0
9080,4.814127e-05,2.986457e-04,6.659154e-05,2.728984e-05,3.939941e-04,0.000012,...,2.047652e-03,0.002605,0.000163,2.316786e-02,3.834179e-02,0.0


In [9]:
inversa_ghosh = modelos.calcular_inversa_ghosh(coeficientes_ghosh)
identidade = pd.DataFrame(
    np.eye(67), index=coeficientes_ghosh.index, columns=coeficientes_ghosh.columns
)
residuo_ghosh = ((identidade - coeficientes_ghosh) @ inversa_ghosh - identidade).abs().to_numpy().max()
print(f"Resíduo máximo de (I − B) @ G − I: {residuo_ghosh:.2e}")
inversa_ghosh

Resíduo máximo de (I − B) @ G − I: 1.44e-15


atividade_destino,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,1.027732,0.024115,1.429796e-03,0.000492,0.002646,0.000839,...,0.000747,0.003307,0.003419,0.000234,0.003882,0.0
0192,0.007038,1.040622,1.314722e-03,0.000122,0.000593,0.000155,...,0.000887,0.004192,0.004288,0.000135,0.003702,0.0
0280,0.034144,0.038603,1.051213e+00,0.000394,0.002122,0.000785,...,0.001312,0.003778,0.005376,0.000467,0.003539,0.0
0580,0.047379,0.030927,1.452268e-03,1.016587,0.034717,0.002401,...,0.001661,0.006639,0.004758,0.000918,0.004615,0.0
0680,0.041332,0.015270,2.051365e-03,0.003447,1.088226,0.010122,...,0.002695,0.004092,0.004790,0.001103,0.007397,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8691,0.000017,0.000008,4.870276e-07,0.000001,0.000010,0.000022,...,0.000003,1.000011,0.001238,0.000001,0.000005,0.0
8692,0.000026,0.000014,1.528568e-06,0.000002,0.000024,0.000008,...,0.000069,0.000086,1.103236,0.000062,0.000061,0.0
9080,0.002570,0.002082,2.763440e-04,0.000273,0.002676,0.000670,...,0.004676,0.005114,0.002738,1.025167,0.042131,0.0


## 6. Alinhamento das intensidades de CO₂

As intensidades carregadas na seção 2 são preparadas para os cálculos. Os setores `S1`–`S67` do artigo seguem a ordem das atividades da MIP. Essa correspondência por posição é a hipótese usada para atribuir os códigos brasileiros às intensidades. Em seguida, verificamos a ordem dos setores e a cobertura dos anos nos parâmetros externos.


In [ ]:
# Os setores S1-S67 do artigo seguem a ordem das atividades da MIP nivel 67.
intensidades_co2 = matriz_coeficientes_co2.copy()
intensidades_co2.index = producao_bruta.index
intensidades_co2.index.name = "atividade"

assert intensidades_co2_exterior.index.equals(producao_bruta.index)
assert inversa_leontief_exterior.index.equals(producao_bruta.index)
assert set(intensidades_co2.columns).issubset(intensidades_co2_exterior.columns)

intensidades_co2


## 7. Tres abordagens de contabilidade de CO2

**Producao territorial.** A emissao fica com a atividade que a gerou no Brasil: `e = gamma_BR * x_BR`. Ela inclui a producao destinada a exportacoes e exclui emissoes ocorridas no exterior.

**Consumo brasileiro.** A conta usa Leontief e exclui a demanda de exportacoes. Ela soma tres componentes: producao brasileira requerida pela demanda domestica, importacoes finais e importacoes intermediarias incorporadas na producao brasileira. Em notacao matricial: `C = gamma_BR_hat L_BR f_dom + gamma_EXT_hat L_EXT f_imp + gamma_EXT_hat L_EXT A_imp L_BR f_dom`.

**Renda no sistema domestico.** A conta usa Ghosh, `R = r_hat G_BR gamma_BR`, e atribui emissoes aos setores que recebem as entradas primarias `r`. Ela ainda nao separa renda brasileira e estrangeira; portanto, nao e uma reproducao internacional completa da responsabilidade baseada em renda de Marques et al. (2012).

Os parametros do exterior (`gamma_EXT` e `L_EXT`) sao lidos dos CSVs em `parameters/exterior_representativo/`. O cenario inicial desses arquivos documenta `gamma_EXT = gamma_BR` e `L_EXT = L_BR`; eles podem ser substituidos por um cenario externo empirico sem modificar as formulas. A conversao de produtos importados para atividades usa a matriz de participacao nacional `D`, isto e, assume a mesma composicao produto-atividade no exterior representativo.

In [ ]:
demanda_final = mip.carregar_demanda_final_nacional_67(tabelas)
demanda_final_domestica = mip.carregar_demanda_final_domestica_67(tabelas)
demanda_final_exportacoes = mip.carregar_demanda_final_exportacoes_67(tabelas)
demanda_final_importada = mip.carregar_demanda_final_importada_67(tabelas)
coeficientes_importados = modelos.calcular_coeficientes_importados_67(tabelas)
insumos_primarios = modelos.calcular_insumos_primarios_ghosh_67(tabelas)

erro_fechamento_demanda = (demanda_final - (demanda_final_domestica + demanda_final_exportacoes)).abs().max()
assert erro_fechamento_demanda < 1e-6
assert (insumos_primarios >= 0).all()
print(f"Demanda final domestica nacional: R$ {demanda_final_domestica.sum():,.0f} milhoes")
print(f"Exportacoes nacionais: R$ {demanda_final_exportacoes.sum():,.0f} milhoes")
print(f"Demanda final atendida por importacoes: R$ {demanda_final_importada.sum():,.0f} milhoes")
print(f"Entradas primarias do sistema de Ghosh: R$ {insumos_primarios.sum():,.0f} milhoes")

In [12]:
resultados_por_ano = {}
totais_por_ano = {}
componentes_consumo_por_ano = {}

for ano, intensidade in intensidades_co2.items():
    emissoes_producao = emissoes.calcular_emissoes_producao(intensidade, producao_bruta)
    # Os parametros do exterior sao lidos de CSV. No cenario inicial, os
    # CSVs documentam a hipotese gamma_exterior = gamma_Brasil e L_exterior = L_Brasil.
    matrizes_consumo = emissoes.calcular_matrizes_emissoes_consumo_com_importacoes(
        intensidade, intensidades_co2_exterior[ano], inversa_leontief,
        inversa_leontief_exterior, demanda_final_domestica,
        demanda_final_importada, coeficientes_importados
    )
    matriz_exportacoes = emissoes.calcular_matriz_emissoes_consumo(
        intensidade, inversa_leontief, demanda_final_exportacoes
    )
    matriz_renda = emissoes.calcular_matriz_emissoes_renda(
        insumos_primarios, inversa_ghosh, intensidade
    )

    resultados_por_ano[ano] = pd.DataFrame({
        "producao": emissoes_producao,
        "consumo": matrizes_consumo["total"].sum(axis=0),
        "renda": matriz_renda.sum(axis=1),
    })
    totais_por_ano[ano] = resultados_por_ano[ano].sum()
    componentes_consumo_por_ano[ano] = pd.Series({
        "producao_domestica_para_consumo": matrizes_consumo["domestica"].to_numpy().sum(),
        "importacoes_finais": matrizes_consumo["importacoes_finais"].to_numpy().sum(),
        "importacoes_intermediarias": matrizes_consumo["importacoes_intermediarias"].to_numpy().sum(),
        "emissoes_brasileiras_para_exportacao": matriz_exportacoes.to_numpy().sum(),
    })

contabilidade_co2 = pd.concat(resultados_por_ano, names=["ano_intensidade", "atividade"])
totais_contabilidade_co2 = pd.DataFrame(totais_por_ano).T
totais_contabilidade_co2.index.name = "ano_intensidade"

componentes_consumo_co2 = pd.DataFrame(componentes_consumo_por_ano).T
componentes_consumo_co2.index.name = "ano_intensidade"

# A conta de consumo agora inclui CO2 externo estimado; por isso nao precisa coincidir
# com producao territorial ou renda calculada no sistema domestico.
display(totais_contabilidade_co2)
componentes_consumo_co2

Maior diferenca entre os totais das tres contas: 2.33e-10 Gg CO2


,producao,consumo,renda
ano_intensidade,,,
2011,692368.19,692368.19,692368.19
2018,677191.93,677191.93,677191.93


In [14]:
# Comparacao das 67 atividades para a intensidade de 2018; altere o ano se necessario.
descricoes_atividades = pd.Series(
    tabelas["14"].iloc[5:72, 1].to_numpy(),
    index=producao_bruta.index,
    name="descricao_atividade",
)

tabela_final_2018 = contabilidade_co2.xs(2018, level="ano_intensidade").join(descricoes_atividades)
tabela_final_2018 = tabela_final_2018[["descricao_atividade", "producao", "consumo", "renda"]]
tabela_final_2018.sort_values("producao", ascending=False)

,descricao_atividade,producao,consumo,renda
atividade,,,,
3500,"Energia elétrica, gás natural e outras utilidades",67795.78,33181.820751,54628.738298
4580,Comércio por atacado e varejo,22015.26,43249.386176,49809.750070
3680,"Água, esgoto e gestão de resíduos",64730.35,26923.469268,46669.057024
4180,Construção,44261.56,64741.803739,32884.923601
6480,"Intermediação financeira, seguros e previdênci...",11492.22,15846.674866,28714.214929
...,...,...,...,...
0791,"Extração de minério de ferro, inclusive benefi...",1055.06,3277.240305,1048.631935
0792,"Extração de minerais metálicos não ferrosos, i...",1610.30,1609.931003,1030.971285
1992,Fabricação de biocombustíveis,1285.23,2325.752810,958.332199


### Gráfico, CSV e proveniência

As barras mostram consumo brasileiro com importações e renda no sistema doméstico; a linha mostra as emissões territoriais. A tabela e o gráfico são exportados em CSV e PNG, junto a um manifesto de proveniência, em `outputs/<execucao_id>/`.

O manifesto identifica as entradas por nome, descrição e SHA-256, além do commit, alterações locais, código salvo e ambiente da execução. O PNG também contém metadados de proveniência; mantenha o CSV junto ao JSON.

In [ ]:
figura_contabilidade, eixo_contabilidade, caminho_proveniencia = execucao.exportar(
    tabela_final_2018,
    arquivos.RAIZ_PROJETO / "outputs",
    nome="contabilidade_co2_2018",
    titulo="Emissões de CO₂ por abordagem de contabilidade (intensidades de 2018)",
)
print(f"PNG, CSV e proveniência salvos em: {caminho_proveniencia.parent}")
figura_contabilidade

## 8. Inventario das tabelas da MIP

Cada chave de `tabelas` corresponde a uma aba do XLS e seu valor e um `pandas.DataFrame`.

In [ ]:
resumo_abas = pd.DataFrame(
    [(aba, *dataframe.shape) for aba, dataframe in tabelas.items()],
    columns=["aba", "linhas", "colunas"],
)
resumo_abas